In [ ]:
import os

for root, dirs, files in os.walk("."):
    if root.startswith("./src") or root in ["./src", "./data", "./reports"]:
        print(root)
        for f in files:
            print("   ", f)

In [ ]:
import os

for root, dirs, files in os.walk("."):
    if root.startswith("./src") or root in ["./src", "./data", "./reports"]:
        print(root)
        for f in files:
            print("   ", f)

In [ ]:
import os, textwrap

os.makedirs("src", exist_ok=True)
os.makedirs("reports", exist_ok=True)
os.makedirs("data", exist_ok=True)

files = {
"requirements.txt": """
yfinance
ta
pandas
numpy
""",

"src/config.py": """
CAPITAL = 100000
RISK_PER_TRADE = 0.01
MAX_POSITIONS = 5
MIN_SCORE = 80

NIFTY500_URL = "https://archives.nseindia.com/content/indices/ind_nifty500list.csv"
NIFTY_SYMBOL = "^NSEI"
""",

"src/data_loader.py": """
import pandas as pd
import yfinance as yf
from config import NIFTY500_URL, NIFTY_SYMBOL

def clean_columns(df):
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df.dropna()

def get_nifty500_symbols():
    df = pd.read_csv(NIFTY500_URL)
    return [s + ".NS" for s in df["Symbol"].dropna().unique().tolist()]

def download_stock(symbol, period="1y", interval="1d"):
    df = yf.download(symbol, period=period, interval=interval, progress=False, auto_adjust=False)
    if df.empty:
        return None
    return clean_columns(df)

def download_nifty():
    df = yf.download(NIFTY_SYMBOL, period="1y", interval="1d", progress=False, auto_adjust=False)
    return clean_columns(df)
""",

"src/indicators.py": """
from ta.trend import EMAIndicator, MACD, ADXIndicator
from ta.momentum import RSIIndicator
from ta.volatility import AverageTrueRange

def add_indicators(df):
    df = df.copy()

    close = df["Close"].squeeze()
    high = df["High"].squeeze()
    low = df["Low"].squeeze()
    volume = df["Volume"].squeeze()

    df["EMA20"] = EMAIndicator(close, 20).ema_indicator()
    df["EMA50"] = EMAIndicator(close, 50).ema_indicator()
    df["EMA200"] = EMAIndicator(close, 200).ema_indicator()
    df["RSI"] = RSIIndicator(close, 14).rsi()

    macd = MACD(close)
    df["MACD"] = macd.macd()
    df["MACD_SIGNAL"] = macd.macd_signal()

    df["ADX"] = ADXIndicator(high, low, close, 14).adx()
    df["ATR"] = AverageTrueRange(high, low, close, 14).average_true_range()

    df["VOL_AVG20"] = volume.rolling(20).mean()
    df["HIGH_20"] = high.rolling(20).max()

    return df.dropna()
""",

"src/scoring.py": """
def relative_strength(stock_df, nifty_df, lookback=60):
    if len(stock_df) < lookback or len(nifty_df) < lookback:
        return 0

    stock_return = stock_df["Close"].iloc[-1] / stock_df["Close"].iloc[-lookback] - 1
    nifty_return = nifty_df["Close"].iloc[-1] / nifty_df["Close"].iloc[-lookback] - 1

    return stock_return - nifty_return

def market_is_bullish(nifty_df):
    latest = nifty_df.iloc[-1]
    return latest["Close"] > latest["EMA200"] and latest["EMA20"] > latest["EMA50"]

def score_stock(df, nifty_df, weekly_ok):
    latest = df.iloc[-1]
    score = 0
    reasons = []

    rs = relative_strength(df, nifty_df)

    if latest["Close"] > latest["EMA200"] and latest["EMA20"] > latest["EMA50"]:
        score += 20
        reasons.append("Daily trend bullish")

    if weekly_ok:
        score += 20
        reasons.append("Weekly trend confirmed")

    if rs > 0.05:
        score += 15
        reasons.append(f"Strong RS vs Nifty ({rs:.1%})")

    if latest["Close"] > df["HIGH_20"].shift(1).iloc[-1]:
        score += 15
        reasons.append("20-day breakout")

    if latest["Volume"] > 1.5 * latest["VOL_AVG20"]:
        score += 10
        reasons.append("Volume breakout")

    if latest["ADX"] > 25:
        score += 10
        reasons.append("ADX strong")

    if 50 <= latest["RSI"] <= 68:
        score += 5
        reasons.append("RSI healthy")

    return score, reasons, rs
""",

"src/scanner.py": """
from data_loader import download_stock
from indicators import add_indicators
from scoring import score_stock
from config import CAPITAL, RISK_PER_TRADE

def weekly_trend(symbol):
    weekly = download_stock(symbol, period="3y", interval="1wk")
    if weekly is None or len(weekly) < 60:
        return False

    weekly = add_indicators(weekly)
    latest = weekly.iloc[-1]

    return latest["Close"] > latest["EMA20"] and latest["EMA20"] > latest["EMA50"]

def analyze_stock(symbol, nifty_df):
    try:
        df = download_stock(symbol)

        if df is None or len(df) < 220:
            return None

        df = add_indicators(df)
        weekly_ok = weekly_trend(symbol)

        score, reasons, rs = score_stock(df, nifty_df, weekly_ok)

        latest = df.iloc[-1]

        entry = latest["Close"]
        stop_loss = entry - (1.5 * latest["ATR"])
        target = entry + (3 * latest["ATR"])

        risk_per_share = entry - stop_loss
        risk_reward = (target - entry) / risk_per_share

        risk_amount = CAPITAL * RISK_PER_TRADE
        quantity = int(risk_amount / risk_per_share)
        capital_required = quantity * entry

        if quantity <= 0:
            return None

        return {
            "Symbol": symbol,
            "Score": round(score, 2),
            "Entry": round(entry, 2),
            "Stop Loss": round(stop_loss, 2),
            "Target": round(target, 2),
            "Quantity": quantity,
            "Capital Required": round(capital_required, 2),
            "Risk Reward": round(risk_reward, 2),
            "Relative Strength": round(rs * 100, 2),
            "RSI": round(latest["RSI"], 2),
            "ADX": round(latest["ADX"], 2),
            "Weekly Trend": weekly_ok,
            "Reasons": ", ".join(reasons)
        }

    except Exception:
        return None
""",

"src/main.py": """
import pandas as pd
from datetime import datetime

from data_loader import get_nifty500_symbols, download_nifty
from indicators import add_indicators
from scoring import market_is_bullish
from scanner import analyze_stock
from config import MIN_SCORE, CAPITAL, MAX_POSITIONS

def main():
    symbols = get_nifty500_symbols()

    nifty_df = download_nifty()
    nifty_df = add_indicators(nifty_df)

    if not market_is_bullish(nifty_df):
        print("Market filter: Nifty trend is weak. No fresh long signals preferred today.")
        return

    results = []

    for i, symbol in enumerate(symbols):
        res = analyze_stock(symbol, nifty_df)

        if res:
            results.append(res)

        if (i + 1) % 50 == 0:
            print(f"Scanned {i + 1} stocks...")

    scanner = pd.DataFrame(results)

    if scanner.empty:
        print("No valid stocks scanned.")
        return

    final = scanner[
        (scanner["Score"] >= MIN_SCORE) &
        (scanner["Risk Reward"] >= 2) &
        (scanner["Capital Required"] <= CAPITAL)
    ].copy()

    final = final.sort_values(
        by=["Score", "Relative Strength"],
        ascending=False
    ).head(MAX_POSITIONS)

    today = datetime.now().strftime("%Y-%m-%d")
    output_file = f"reports/signals_{today}.csv"
    final.to_csv(output_file, index=False)

    print("\\n🏆 TOP NIFTY 500 SWING SIGNALS")
    print("--------------------------------")

    if final.empty:
        print("No high-quality signals today.")
    else:
        print(final.to_string(index=False))

    print(f"\\nSaved report: {output_file}")

if __name__ == "__main__":
    main()
"""
}

for path, content in files.items():
    with open(path, "w") as f:
        f.write(textwrap.dedent(content).strip() + "\n")

print("Professional project created successfully.")

Professional project created successfully.


In [ ]:
!python src/main.py

Failed to get ticker '^NSEI' reason: Failed to perform, curl: (60) SSL: no alternative certificate subject name matches target hostname 'fc.yahoo.com'. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
Market filter: Nifty trend is weak. No fresh long signals preferred today.


In [ ]:
!pip install -q yfinance ta pandas numpy

In [ ]:
!python src/main.py

Failed to get ticker '^NSEI' reason: Failed to perform, curl: (60) SSL: no alternative certificate subject name matches target hostname 'fc.yahoo.com'. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.

1 Failed download:
['^NSEI']: CertificateVerifyError("Failed to perform, curl: (60) SSL: no alternative certificate subject name matches target hostname 'fc.yahoo.com'. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.")
Traceback (most recent call last):
  File "/content/src/main.py", line 63, in <module>
    main()
  File "/content/src/main.py", line 14, in main
    nifty_df = add_indicators(nifty_df)
               ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/src/indicators.py", line 22, in add_indicators
    df["ADX"] = ADXIndicator(high, low, close, 14).adx()
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ta/trend.py", line 716, in __init__
    self._run()
  File "/usr/local/lib/python3

In [ ]:
%%writefile src/data_loader.py

import pandas as pd
import yfinance as yf
from config import NIFTY500_URL, NIFTY_SYMBOLS


def clean_columns(df):
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df.dropna()


def get_nifty500_symbols():
    df = pd.read_csv(NIFTY500_URL)
    return [s + ".NS" for s in df["Symbol"].dropna().unique().tolist()]


def download_stock(symbol, period="1y", interval="1d"):
    df = yf.download(symbol, period=period, interval=interval, progress=False, auto_adjust=False)

    if df.empty:
        return None

    return clean_columns(df)


def download_nifty():
    for symbol in NIFTY_SYMBOLS:
        df = yf.download(symbol, period="1y", interval="1d", progress=False, auto_adjust=False)

        if not df.empty:
            print(f"Benchmark loaded: {symbol}")
            return clean_columns(df)

    return None

Overwriting src/data_loader.py


In [ ]:
!python src/main.py

Market filter: Nifty trend is weak. No fresh long signals preferred today.


In [ ]:
!sed -i 's/return$/# return/' src/main.py

In [ ]:
!python src/main.py

Market filter: Nifty trend is weak. No fresh long signals preferred today.
Scanned 50 stocks...
Scanned 100 stocks...
Scanned 150 stocks...
Scanned 200 stocks...
Scanned 250 stocks...
Scanned 300 stocks...
Scanned 350 stocks...
Scanned 400 stocks...
Scanned 450 stocks...
Scanned 500 stocks...

🏆 TOP NIFTY 500 SWING SIGNALS
--------------------------------
No high-quality signals today.

Saved report: reports/signals_2026-07-08.csv


In [ ]:
import pandas as pd

df = pd.read_csv("reports/signals_2026-07-08.csv")
df

,Symbol,Score,Entry,Stop Loss,Target,Quantity,Capital Required,Risk Reward,Relative Strength,RSI,ADX,Weekly Trend,Reasons


In [ ]:
# Patch main.py to save ALL scanned stocks for debugging
!python - <<'PY'
path = "src/main.py"
text = open(path).read()
text = text.replace(
    'final_signals.to_csv(output_file, index=False)',
    'scanner_df.to_csv("reports/debug_all_stocks.csv", index=False)\\n    final_signals.to_csv(output_file, index=False)'
)
open(path, "w").write(text)
PY

!python src/main.py

/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')


NameError: name 'PY' is not defined

In [ ]:
%%writefile src/main.py

import pandas as pd
from datetime import datetime

from config import MIN_SCORE, CAPITAL, MAX_POSITIONS
from data_loader import get_nifty500_symbols, download_benchmark
from indicators import add_indicators
from market_filter import market_is_bullish
from scanner import analyze_stock
from telegram_bot import format_signals_message


def main():
    print("Starting Nifty 500 Swing Scanner v1.0")

    symbols = get_nifty500_symbols()
    print(f"Loaded {len(symbols)} Nifty 500 symbols")

    benchmark_df = download_benchmark()

    if benchmark_df is None or len(benchmark_df) < 220:
        print("Benchmark data failed. Please rerun later.")
        return

    benchmark_df = add_indicators(benchmark_df)

    if not market_is_bullish(benchmark_df):
        print("Market filter: Nifty trend is weak.")
        print("Scanner will continue in WATCHLIST mode for debugging.")

    results = []

    for i, symbol in enumerate(symbols):
        result = analyze_stock(symbol, benchmark_df)

        if result is not None:
            results.append(result)

        if (i + 1) % 50 == 0:
            print(f"Scanned {i + 1} stocks...")

    scanner_df = pd.DataFrame(results)

    if scanner_df.empty:
        print("No valid stocks scanned.")
        return

    scanner_df.to_csv("reports/debug_all_stocks.csv", index=False)

    final_signals = scanner_df[
        (scanner_df["Score"] >= MIN_SCORE) &
        (scanner_df["Risk Reward"] >= 2) &
        (scanner_df["Capital Required"] <= CAPITAL)
    ].copy()

    final_signals = final_signals.sort_values(
        by=["Score", "Relative Strength %"],
        ascending=False
    ).head(MAX_POSITIONS)

    today = datetime.now().strftime("%Y-%m-%d")
    output_file = f"reports/signals_{today}.csv"
    final_signals.to_csv(output_file, index=False)

    print("\n🏆 TOP NIFTY 500 SWING SIGNALS")
    print("--------------------------------")

    if final_signals.empty:
        print("No high-quality signals today.")
    else:
        print(final_signals.to_string(index=False))

    print(f"\nSaved final report: {output_file}")
    print("Saved debug report: reports/debug_all_stocks.csv")


if __name__ == "__main__":
    main()

Overwriting src/main.py


In [ ]:
!python src/main.py

Traceback (most recent call last):
  File "/content/src/main.py", line 6, in <module>
    from data_loader import get_nifty500_symbols, download_benchmark
ImportError: cannot import name 'download_benchmark' from 'data_loader' (/content/src/data_loader.py)


In [ ]:
import pandas as pd

debug = pd.read_csv("reports/debug_all_stocks.csv")
debug.sort_values("Score", ascending=False).head(20)

FileNotFoundError: [Errno 2] No such file or directory: 'reports/debug_all_stocks.csv'

In [ ]:
!python src/main.py

Traceback (most recent call last):
  File "/content/src/main.py", line 6, in <module>
    from data_loader import get_nifty500_symbols, download_benchmark
ImportError: cannot import name 'download_benchmark' from 'data_loader' (/content/src/data_loader.py)


In [ ]:
%%writefile src/data_loader.py

import pandas as pd
import yfinance as yf
from config import NIFTY500_URL, NIFTY_SYMBOLS

def clean_columns(df):
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df.dropna()

def get_nifty500_symbols():
    df = pd.read_csv(NIFTY500_URL)
    symbols = df["Symbol"].dropna().unique().tolist()
    return [s + ".NS" for s in symbols]

def download_stock(symbol, period="1y", interval="1d"):
    df = yf.download(
        symbol,
        period=period,
        interval=interval,
        progress=False,
        auto_adjust=False
    )

    if df.empty:
        return None

    return clean_columns(df)

def download_benchmark(period="1y", interval="1d"):
    """
    Downloads the benchmark (Nifty 50 or fallback ETF).
    """
    for symbol in NIFTY_SYMBOLS:
        try:
            df = yf.download(
                symbol,
                period=period,
                interval=interval,
                progress=False,
                auto_adjust=False
            )

            if not df.empty:
                print(f"Benchmark loaded: {symbol}")
                return clean_columns(df)

        except Exception:
            continue

    return None

Overwriting src/data_loader.py


In [ ]:
!python src/main.py

Traceback (most recent call last):
  File "/content/src/main.py", line 6, in <module>
    from data_loader import get_nifty500_symbols, download_benchmark
  File "/content/src/data_loader.py", line 4, in <module>
    from config import NIFTY500_URL, NIFTY_SYMBOLS
ImportError: cannot import name 'NIFTY_SYMBOLS' from 'config' (/content/src/config.py). Did you mean: 'NIFTY_SYMBOL'?


In [ ]:
%%writefile src/config.py

CAPITAL = 100000
RISK_PER_TRADE = 0.01
MAX_POSITIONS = 5
MIN_SCORE = 80

# NSE Nifty 500 list
NIFTY500_URL = "https://archives.nseindia.com/content/indices/ind_nifty500list.csv"

# Benchmark symbols (fallback if one fails)
NIFTY_SYMBOLS = [
    "^NSEI",
    "NIFTYBEES.NS"
]

# Data periods
DAILY_PERIOD = "1y"
WEEKLY_PERIOD = "3y"

# Risk Management
ATR_MULTIPLIER_STOP = 1.5
RISK_REWARD_TARGET = 2.0

Overwriting src/config.py


In [ ]:
!python src/main.py

Traceback (most recent call last):
  File "/content/src/main.py", line 8, in <module>
    from market_filter import market_is_bullish
ModuleNotFoundError: No module named 'market_filter'


In [ ]:
%%writefile src/market_filter.py

def market_is_bullish(nifty_df):
    latest = nifty_df.iloc[-1]

    return (
        latest["Close"] > latest["EMA200"] and
        latest["EMA20"] > latest["EMA50"]
    )

Writing src/market_filter.py


In [ ]:
!python src/main.py

Traceback (most recent call last):
  File "/content/src/main.py", line 10, in <module>
    from telegram_bot import format_signals_message
ModuleNotFoundError: No module named 'telegram_bot'


In [ ]:
results = []

for i, symbol in enumerate(symbols):
    res = analyze_stock(symbol, nifty_df)

    if res:
        results.append(res)

    if (i + 1) % 50 == 0:
        print(f"Scanned {i + 1} stocks...")

scanner = pd.DataFrame(results)

print("Stocks that passed analysis:", len(scanner))

if scanner.empty:
    print("No stocks passed analysis.")
else:
    display(scanner.sort_values("Score", ascending=False).head(10))

NameError: name 'symbols' is not defined

In [ ]:
# Recreate required variables

symbols = get_nifty500_symbols()

nifty_df = get_benchmark()

if nifty_df is None:
    raise Exception("Benchmark download failed.")

nifty_df = add_indicators(nifty_df)

print("Benchmark ready")
print("Total symbols:", len(symbols))

NameError: name 'get_nifty500_symbols' is not defined

In [ ]:
!pip install -q yfinance ta pandas numpy requests

  Preparing metadata (setup.py) ... done


In [ ]:
BOT_TOKEN = "8817443910:AAFki_yHbWDzVu6ozzC6d4k9c8heyKhvuZI"
CHAT_ID = "8682661998"

CAPITAL = 100000
RISK_PER_TRADE = 0.01
MAX_POSITIONS = 5
MIN_SCORE = 65

print("Ready")

Ready


In [ ]:
import pandas as pd
import yfinance as yf
import requests
from datetime import datetime

from ta.trend import EMAIndicator, MACD, ADXIndicator
from ta.momentum import RSIIndicator
from ta.volatility import AverageTrueRange

NIFTY500_URL = "https://archives.nseindia.com/content/indices/ind_nifty500list.csv"
BENCHMARKS = ["^NSEI", "NIFTYBEES.NS"]

def clean(df):
    if df is None or df.empty:
        return None
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df.dropna()
    return df if not df.empty else None

def download(symbol, period="1y", interval="1d"):
    try:
        df = yf.download(symbol, period=period, interval=interval, progress=False, auto_adjust=False)
        return clean(df)
    except:
        return None

def get_symbols():
    df = pd.read_csv(NIFTY500_URL)
    return [s + ".NS" for s in df["Symbol"].dropna().unique().tolist()]

def get_benchmark():
    for s in BENCHMARKS:
        df = download(s)
        if df is not None and len(df) > 220:
            print("Benchmark loaded:", s)
            return df
    return None

def add_indicators(df):
    try:
        df = df.copy()
        close = df["Close"].squeeze()
        high = df["High"].squeeze()
        low = df["Low"].squeeze()
        volume = df["Volume"].squeeze()

        df["EMA20"] = EMAIndicator(close=close, window=20).ema_indicator()
        df["EMA50"] = EMAIndicator(close=close, window=50).ema_indicator()
        df["EMA200"] = EMAIndicator(close=close, window=200).ema_indicator()
        df["RSI"] = RSIIndicator(close=close, window=14).rsi()

        macd = MACD(close=close)
        df["MACD"] = macd.macd()
        df["MACD_SIGNAL"] = macd.macd_signal()

        df["ADX"] = ADXIndicator(high=high, low=low, close=close, window=14).adx()
        df["ATR"] = AverageTrueRange(high=high, low=low, close=close, window=14).average_true_range()

        df["VOL_AVG20"] = volume.rolling(20).mean()
        df["HIGH_20"] = high.rolling(20).max()

        return df.dropna()
    except:
        return None

def relative_strength(stock_df, nifty_df, lookback=60):
    try:
        stock_return = stock_df["Close"].iloc[-1] / stock_df["Close"].iloc[-lookback] - 1
        nifty_return = nifty_df["Close"].iloc[-1] / nifty_df["Close"].iloc[-lookback] - 1
        return stock_return - nifty_return
    except:
        return 0

def weekly_trend(symbol):
    try:
        weekly = download(symbol, period="3y", interval="1wk")
        if weekly is None or len(weekly) < 80:
            return False

        close = weekly["Close"].squeeze()
        weekly["EMA20"] = EMAIndicator(close=close, window=20).ema_indicator()
        weekly["EMA50"] = EMAIndicator(close=close, window=50).ema_indicator()
        weekly = weekly.dropna()

        if weekly.empty:
            return False

        latest = weekly.iloc[-1]
        return latest["Close"] > latest["EMA20"] and latest["EMA20"] > latest["EMA50"]
    except:
        return False

def analyze_stock(symbol, nifty_df):
    try:
        df = download(symbol)
        if df is None or len(df) < 220:
            return None

        df = add_indicators(df)
        if df is None or df.empty:
            return None

        latest = df.iloc[-1]
        score = 0
        reasons = []

        rs = relative_strength(df, nifty_df)
        weekly_ok = weekly_trend(symbol)

        if latest["Close"] > latest["EMA200"] and latest["EMA20"] > latest["EMA50"]:
            score += 20
            reasons.append("Daily bullish")

        if weekly_ok:
            score += 20
            reasons.append("Weekly bullish")

        if rs > 0.05:
            score += 15
            reasons.append(f"RS +{rs:.1%}")

        if latest["Close"] > df["HIGH_20"].shift(1).iloc[-1]:
            score += 15
            reasons.append("Breakout")

        if latest["Volume"] > 1.5 * latest["VOL_AVG20"]:
            score += 10
            reasons.append("Volume breakout")

        if latest["ADX"] > 25:
            score += 10
            reasons.append("ADX strong")

        if 50 <= latest["RSI"] <= 68:
            score += 10
            reasons.append("RSI healthy")

        entry = float(latest["Close"])
        stop = entry - 1.5 * float(latest["ATR"])
        risk = entry - stop
        target = entry + 2 * risk

        if risk <= 0:
            return None

        qty = int((CAPITAL * RISK_PER_TRADE) / risk)
        capital_required = qty * entry

        if qty <= 0:
            return None

        return {
            "Symbol": symbol,
            "Score": score,
            "Entry": round(entry, 2),
            "Stop Loss": round(stop, 2),
            "Target": round(target, 2),
            "Quantity": qty,
            "Capital Required": round(capital_required, 2),
            "RS %": round(rs * 100, 2),
            "RSI": round(float(latest["RSI"]), 2),
            "ADX": round(float(latest["ADX"]), 2),
            "Reasons": ", ".join(reasons)
        }
    except:
        return None

print("Core scanner loaded")

Core scanner loaded


In [ ]:
symbols = get_symbols()
nifty_df = get_benchmark()

if nifty_df is None:
    raise Exception("Benchmark failed. Run again after 1 minute.")

nifty_df = add_indicators(nifty_df)

results = []

for i, symbol in enumerate(symbols):
    res = analyze_stock(symbol, nifty_df)
    if res:
        results.append(res)

    if (i + 1) % 50 == 0:
        print(f"Scanned {i + 1} stocks...")

scanner = pd.DataFrame(results)
print("Stocks analyzed:", len(scanner))

final = scanner[
    (scanner["Score"] >= MIN_SCORE) &
    (scanner["Capital Required"] <= CAPITAL)
].copy()

final = final.sort_values(["Score", "RS %"], ascending=False).head(MAX_POSITIONS)

display(final)

msg = "🏆 NIFTY 500 SWING SIGNALS\n\n"

if final.empty:
    msg += "No high-quality setups today."
else:
    for _, row in final.iterrows():
        msg += f"""📈 {row['Symbol']}
Score: {row['Score']}/100
Entry: ₹{row['Entry']}
SL: ₹{row['Stop Loss']}
Target: ₹{row['Target']}
Qty: {row['Quantity']}
RS: {row['RS %']}%
Reasons: {row['Reasons']}

"""

url = f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage"
payload = {"chat_id": CHAT_ID, "text": msg}
r = requests.post(url, json=payload)

print(r.status_code)
print(r.text[:300])

today = datetime.now().strftime("%Y-%m-%d")
final.to_csv(f"signals_{today}.csv", index=False)

print("Done")

Benchmark loaded: ^NSEI
Scanned 50 stocks...
Scanned 100 stocks...
Scanned 150 stocks...
Scanned 200 stocks...
Scanned 250 stocks...
Scanned 300 stocks...
Scanned 350 stocks...
Scanned 400 stocks...
Scanned 450 stocks...
Scanned 500 stocks...
Stocks analyzed: 477


,Symbol,Score,Entry,Stop Loss,Target,Quantity,Capital Required,RS %,RSI,ADX,Reasons
245,JBCHEPHARM.NS,75,2435.80,2361.68,2584.04,13,31665.40,0,81.09,35.56,"Daily bullish, Weekly bullish, Breakout, Volum..."
156,EXIDEIND.NS,70,421.35,403.68,456.69,56,23595.60,0,67.87,27.07,"Daily bullish, Weekly bullish, Volume breakout..."
256,J&KBANK.NS,70,165.96,158.14,181.60,127,21076.92,0,66.69,32.25,"Daily bullish, Weekly bullish, Volume breakout..."
293,LUPIN.NS,70,2459.00,2387.26,2602.48,13,31967.00,0,66.01,31.61,"Daily bullish, Weekly bullish, Volume breakout..."
406,SUNPHARMA.NS,70,1888.20,1842.62,1979.37,21,39652.20,0,61.20,34.01,"Daily bullish, Weekly bullish, Volume breakout..."


200
{"ok":true,"result":{"message_id":5,"from":{"id":8817443910,"is_bot":true,"first_name":"Nifty 500 Swing Scanner","username":"nifty500swing_bot"},"chat":{"id":8682661998,"first_name":"Madhu Mohan","type":"private"},"date":1783523799,"text":"\ud83c\udfc6 NIFTY 500 SWING SIGNALS\n\n\ud83d\udcc8 JBCHEPH
Done


In [ ]:
def relative_strength(stock_df, nifty_df, lookback=60):
    try:
        stock_df = stock_df.dropna()
        nifty_df = nifty_df.dropna()

        if len(stock_df) < lookback or len(nifty_df) < lookback:
            return 0

        stock_return = (stock_df["Close"].iloc[-1] / stock_df["Close"].iloc[-lookback]) - 1
        nifty_return = (nifty_df["Close"].iloc[-1] / nifty_df["Close"].iloc[-lookback]) - 1

        rs = stock_return - nifty_return
        return float(rs)

    except Exception as e:
        print("RS error:", e)
        return 0

In [ ]:
results = []

for i, symbol in enumerate(symbols):
    res = analyze_stock(symbol, nifty_df)
    if res:
        results.append(res)

    if (i + 1) % 50 == 0:
        print(f"Scanned {i + 1} stocks...")

scanner = pd.DataFrame(results)
print("Stocks analyzed:", len(scanner))

final = scanner[
    (scanner["Score"] >= MIN_SCORE) &
    (scanner["Capital Required"] <= CAPITAL)
].copy()

final = final.sort_values(["Score", "RS %"], ascending=False).head(MAX_POSITIONS)

display(final)

msg = "🏆 NIFTY 500 SWING SIGNALS\n\n"

if final.empty:
    msg += "No high-quality setups today."
else:
    for _, row in final.iterrows():
        msg += f"""📈 {row['Symbol']}
Score: {row['Score']}/100
Entry: ₹{row['Entry']}
SL: ₹{row['Stop Loss']}
Target: ₹{row['Target']}
Qty: {row['Quantity']}
RS: {row['RS %']}%
Reasons: {row['Reasons']}

"""

url = f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage"
payload =

SyntaxError: invalid syntax (4209808701.py, line 41)

In [ ]:
results = []

for i, symbol in enumerate(symbols):
    res = analyze_stock(symbol, nifty_df)
    if res:
        results.append(res)

    if (i + 1) % 50 == 0:
        print("Scanned", i + 1, "stocks...")

scanner = pd.DataFrame(results)
print("Stocks analyzed:", len(scanner))

final = scanner[
    (scanner["Score"] >= MIN_SCORE) &
    (scanner["Capital Required"] <= CAPITAL)
].copy()

final = final.sort_values(["Score", "RS %"], ascending=False).head(MAX_POSITIONS)

display(final)

msg = "🏆 NIFTY 500 SWING SIGNALS\n\n"

if final.empty:
    msg = msg + "No high-quality setups today."
else:
    for _, row in final.iterrows():
        msg = msg + (
            "📈 " + str(row["Symbol"]) + "\n"
            "Score: " + str(row["Score"]) + "/100\n"
            "Entry: ₹" + str(row["Entry"]) + "\n"
            "SL: ₹" + str(row["Stop Loss"]) + "\n"
            "Target: ₹" + str(row["Target"]) + "\n"
            "Qty: " + str(row["Quantity"]) + "\n"
            "RS: " + str(row["RS %"]) + "%\n"
            "Reasons: " + str(row["Reasons"]) + "\n\n"
        )

url = "https://api.telegram.org/bot" + BOT_TOKEN + "/sendMessage"
payload = {"chat_id": CHAT_ID, "text": msg}
r = requests.post(url, json=payload)

print(r.status_code)
print(r.text[:300])

today = datetime.now().strftime("%Y-%m-%d")
final.to_csv("signals_" + today + ".csv", index=False)

print("Done")

Scanned 50 stocks...
Scanned 100 stocks...
Scanned 150 stocks...
Scanned 200 stocks...
Scanned 250 stocks...
Scanned 300 stocks...
Scanned 350 stocks...
Scanned 400 stocks...
Scanned 450 stocks...
Scanned 500 stocks...
Stocks analyzed: 477


,Symbol,Score,Entry,Stop Loss,Target,Quantity,Capital Required,RS %,RSI,ADX,Reasons
245,JBCHEPHARM.NS,75,2435.80,2361.68,2584.04,13,31665.40,0,81.09,35.56,"Daily bullish, Weekly bullish, Breakout, Volum..."
156,EXIDEIND.NS,70,421.35,403.68,456.69,56,23595.60,0,67.87,27.07,"Daily bullish, Weekly bullish, Volume breakout..."
256,J&KBANK.NS,70,165.96,158.14,181.60,127,21076.92,0,66.69,32.25,"Daily bullish, Weekly bullish, Volume breakout..."
293,LUPIN.NS,70,2459.00,2387.26,2602.48,13,31967.00,0,66.01,31.61,"Daily bullish, Weekly bullish, Volume breakout..."
406,SUNPHARMA.NS,70,1888.20,1842.62,1979.37,21,39652.20,0,61.20,34.01,"Daily bullish, Weekly bullish, Volume breakout..."


200
{"ok":true,"result":{"message_id":7,"from":{"id":8817443910,"is_bot":true,"first_name":"Nifty 500 Swing Scanner","username":"nifty500swing_bot"},"chat":{"id":8682661998,"first_name":"Madhu Mohan","type":"private"},"date":1783524528,"text":"\ud83c\udfc6 NIFTY 500 SWING SIGNALS\n\n\ud83d\udcc8 JBCHEPH
Done
